In [ ]:
REPO_URL = "https://github.com/huyvanzzz/finetune-InternVL2.git"
BRANCH = "main"
PROJECT_DIR = "/kaggle/working/finetune-InternVL2"
CONFIG_PATH = "internvl_config.yaml"
OUTPUT_JSON = "results/resume_smoke.json"


In [ ]:
import os, subprocess, pathlib
if not pathlib.Path(PROJECT_DIR).exists():
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
print("Current repo:", os.getcwd())


In [ ]:
!pip uninstall -y torch torchvision torchaudio triton flash-attn || true
!pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu126 torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1
!pip install -q --no-cache-dir bitsandbytes peft transformers==4.46.2 datasets accelerate timm evaluate rouge_score scikit-learn safetensors huggingface_hub sentencepiece mlflow pyyaml pillow tqdm


In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('capability:', torch.cuda.get_device_capability(0))
    x = torch.zeros(1, device='cuda', dtype=torch.float16)
    print('cuda fp16 tensor ok:', x.dtype)
!python -m py_compile train.py wad_dataset.py qformer_bridge.py scripts/test_resume_equivalence.py scripts/test_resume_real_pipeline_smoke.py scripts/prepare_qformer.py scripts/smoke_qformer_bridge.py


In [ ]:
import subprocess
subprocess.run(["python", "build_frame_index.py"], input="n\n", text=True, check=True)


In [ ]:
!python scripts/prepare_qformer.py --config {CONFIG_PATH}


In [ ]:
!python scripts/smoke_qformer_bridge.py --config {CONFIG_PATH}


In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/test_resume_real_pipeline_smoke.py",
    "--config", CONFIG_PATH,
    "--output_json", OUTPUT_JSON,
], check=True)


In [ ]:
import json
from pathlib import Path

report_path = Path(OUTPUT_JSON)
with open(report_path, 'r', encoding='utf-8') as f:
    report = json.load(f)

print('=== RESUME SMOKE SUMMARY ===')
print('baseline_run_dir:', report['baseline_run_dir'])
print('resumed_run_dir :', report['resumed_run_dir'])
print('step_checkpoint :', report['step_checkpoint_dir'])
print('final_hash_equal:', report['final_hash_equal'])
print('baseline_final_hash:', report['baseline_final_hash'])
print('resumed_final_hash :', report['resumed_final_hash'])
print('baseline sample-id logs:', len(report['baseline_logs']['sample_id_logs']))
print('baseline rng logs      :', len(report['baseline_logs']['rng_logs']))
print('resumed sample-id logs :', len(report['resumed_logs']['sample_id_logs']))
print('resumed rng logs       :', len(report['resumed_logs']['rng_logs']))

print('\n=== BASELINE SAMPLE LOGS (first 10) ===')
for line in report['baseline_logs']['sample_id_logs'][:10]:
    print(line)

print('\n=== RESUMED SAMPLE LOGS (first 10) ===')
for line in report['resumed_logs']['sample_id_logs'][:10]:
    print(line)
